# 04 — DPO Preference Refinement

After SFT, use **Direct Preference Optimization (DPO)** to make Qwen prefer
technically correct, precise RF design answers over vague or incorrect ones.

## How DPO works here
For each RF question we generate two model responses:
- **chosen**: correct answer (with equations, proper units, realistic values)
- **rejected**: plausible-sounding but wrong answer (wrong equations, wrong numbers)

DPO trains the model to assign higher probability to `chosen` without needing
a separate reward model (simpler and more stable than PPO/RLHF).

In [ ]:
!pip install trl transformers peft bitsandbytes accelerate

In [ ]:
import json, random, requests
from pathlib import Path
from tqdm import tqdm
import torch

PROC_DIR  = Path('../data/processed')
SYNTH_DIR = Path('../data/synthetic')
ADAPTER_DIR = Path('../data/models/qwen_rf_lora/adapter')
DPO_MODEL_DIR = Path('../data/models/qwen_rf_dpo')
DPO_MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'

OLLAMA_HOST = "http://localhost:11434"
OLLAMA_MODEL = "qwen3:8b"

## 1. Generate DPO Preference Pairs via Ollama (local qwen3:8b)

In [ ]:
# RF questions where wrong answers are dangerous (wrong designs, wrong math)
DPO_PROMPTS = [
    'What is the optimal transistor finger width for a 28 GHz CMOS LNA in 65nm to minimize noise figure?',
    'Calculate the free-space path loss for a Ka-band (28 GHz) LEO satellite at 550 km altitude.',
    'What inductance value should I use for the source degeneration inductor in a 28 GHz LNA to achieve 50Ω input match?',
    'Explain the tradeoff between LNA gain and noise figure in a cascaded phased array receiver.',
    'What is the resonant frequency of a microstrip patch antenna with L=2.6mm, W=3.5mm on εr=3.55, h=0.508mm?',
    'How does radiation affect the noise figure of a CMOS LNA in LEO orbit over 5 years?',
    'Write the GLayout Python function signature to create a 28 GHz microstrip inductor.',
    'What phase shifter architecture minimizes NF degradation in a Ka-band phased array receiver?',
    'Calculate Friis noise figure for: LNA (NF=2.5dB, G=18dB) → Mixer (NF=8dB, G=-6dB) → IF Amp (NF=5dB).',
    'What is the minimum number of antenna elements in a Ka-band phased array to achieve 30 dBi gain?',
]

SYSTEM_CORRECT = (
    "You are an RF IC design expert. Generate a technically precise answer with "
    "correct equations and realistic values. /no_think"
)
SYSTEM_WRONG = (
    "You are generating INCORRECT RF design answers for training data purposes. "
    "Introduce subtle errors: wrong equations, off-by-10x numerical values, confused concepts, "
    "or wrong units. Sound confident but be technically wrong. /no_think"
)

def _ollama_call(system: str, user: str, max_tokens: int = 1024) -> str:
    prompt = (
        f"<|im_start|>system\n{system}<|im_end|>\n"
        f"<|im_start|>user\n{user}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.7, "num_predict": max_tokens},
    }
    try:
        resp = requests.post(f"{OLLAMA_HOST}/api/generate", json=payload, timeout=120)
        resp.raise_for_status()
        return resp.json().get("response", "")
    except Exception as e:
        print(f"  Ollama error: {e}")
        return ""

def _ollama_available():
    try:
        r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=3)
        return r.status_code == 200
    except Exception:
        return False

def generate_dpo_pair_ollama(question: str) -> dict:
    chosen = _ollama_call(SYSTEM_CORRECT, question)
    wrong_prompt = (
        f"Write a plausible-sounding but technically INCORRECT answer to:\n{question}\n"
        "Introduce subtle errors in equations or numerical values. Output only the answer."
    )
    rejected = _ollama_call(SYSTEM_WRONG, wrong_prompt)
    return {'prompt': question, 'chosen': chosen, 'rejected': rejected}

dpo_pairs = []
dpo_path = SYNTH_DIR / 'dpo_rf_pairs.jsonl'

if _ollama_available():
    print(f"Ollama available — generating DPO pairs with {OLLAMA_MODEL}")
    for q in tqdm(DPO_PROMPTS, desc='Generating DPO pairs'):
        pair = generate_dpo_pair_ollama(q)
        if pair['chosen'] and pair['rejected']:
            dpo_pairs.append(pair)
        print(f'  Q: {q[:60]}...')
    with open(dpo_path, 'w') as f:
        for p in dpo_pairs:
            f.write(json.dumps(p) + '\n')
    print(f'DPO pairs saved: {len(dpo_pairs)} -> {dpo_path}')
else:
    print(f"Ollama not running at {OLLAMA_HOST}. Start with: ollama serve && ollama pull {OLLAMA_MODEL}")
    print("Skipping DPO pair generation — load existing pairs from file if available.")

## 2. Format DPO Dataset in TRL Format

In [ ]:
from datasets import Dataset

SYSTEM_MSG = 'You are an expert RF IC and antenna engineer specializing in Ka-band CMOS chip design for satellite communications.'

def to_dpo_format(pair: dict) -> dict:
    prompt_msgs = [
        {'role': 'system', 'content': SYSTEM_MSG},
        {'role': 'user',   'content': pair['prompt']},
    ]
    return {
        'prompt':   prompt_msgs,
        'chosen':   [{'role': 'assistant', 'content': pair['chosen']}],
        'rejected': [{'role': 'assistant', 'content': pair['rejected']}],
    }

# Load all DPO pairs (handcrafted + generated)
all_pairs = []
dpo_path = SYNTH_DIR / 'dpo_rf_pairs.jsonl'
if dpo_path.exists():
    with open(dpo_path) as f:
        all_pairs = [json.loads(l) for l in f if l.strip()]

dpo_records = [to_dpo_format(p) for p in all_pairs]
random.shuffle(dpo_records)
split = max(1, int(0.9 * len(dpo_records)))
dpo_train = Dataset.from_list(dpo_records[:split])
dpo_val   = Dataset.from_list(dpo_records[split:])

print(f'DPO train: {len(dpo_train)} | val: {len(dpo_val)}')

## 3. DPO Training with TRL

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import DPOTrainer, DPOConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'  # DPO needs left-padding

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)

# Load SFT-trained model as starting point
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config,
    device_map='auto', trust_remote_code=True, use_cache=False,
)
base_model = prepare_model_for_kbit_training(base_model)

# Load SFT LoRA adapter
if ADAPTER_DIR.exists():
    model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR), is_trainable=True)
    print('Loaded SFT adapter — starting DPO from SFT checkpoint')
else:
    # Fresh LoRA if SFT adapter not available
    lora_cfg = LoraConfig(r=32, lora_alpha=64, target_modules=['q_proj','v_proj'], task_type='CAUSAL_LM')
    model = get_peft_model(base_model, lora_cfg)
    print('No SFT adapter found — applying fresh LoRA for DPO')

dpo_config = DPOConfig(
    output_dir=str(DPO_MODEL_DIR / 'checkpoints'),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-5,
    beta=0.1,                    # DPO regularization strength (0.1 = standard)
    max_length=2048,
    max_prompt_length=512,
    bf16=True,
    logging_steps=5,
    save_strategy='epoch',
    report_to='wandb',
    run_name='qwen_rf_dpo',
    remove_unused_columns=False,
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,   # None = use the SFT model itself as reference (implicit DPO)
    args=dpo_config,
    train_dataset=dpo_train,
    eval_dataset=dpo_val,
    tokenizer=tokenizer,
)

print('Starting DPO training...')
dpo_trainer.train()

In [ ]:
dpo_adapter_dir = DPO_MODEL_DIR / 'adapter'
model.save_pretrained(str(dpo_adapter_dir))
tokenizer.save_pretrained(str(dpo_adapter_dir))
print(f'DPO adapter saved to {dpo_adapter_dir}')